In [3]:
from __future__ import annotations

import argparse
import csv
import json
import sys
from collections import Counter
from pathlib import Path


def configure_csv_limit() -> None:
    limit = sys.maxsize

    while True:
        try:
            csv.field_size_limit(limit)
            return
        except OverflowError:
            limit //= 10


def iter_rows(paths: list[Path]):
    for path in paths:
        with path.open(
            "r",
            encoding="utf-8-sig",
            newline="",
        ) as stream:
            reader = csv.DictReader(stream)

            for row in reader:
                yield path, row



In [8]:
configure_csv_limit()

# parser = argparse.ArgumentParser()
# parser.add_argument(
#     "--input",
#     type=Path,
#     default=Path("../data/raw/microservices"),
# )
# args = parser.parse_args([])

input_path = Path("../data/raw/microservices")

paths = sorted(input_path.rglob("*.csv"))

if not paths:
    raise FileNotFoundError(
        f"Nenhum CSV encontrado em {input_path}"
    )

# Bit 1: possui ERROR
# Bit 2: possui WARN
# Bit 4: possui outro nível diferente de INFO
trace_flags: dict[str, int] = {}

total_rows = 0
rows_without_trace = 0
level_counts = Counter()

for path, row in iter_rows(paths):
    total_rows += 1

    if total_rows % 100_000 == 0:
        print(
            f"[progresso] {total_rows:,} linhas processadas",
            flush=True,
        )

    trace_id = (row.get("tc_trace_id") or "").strip()
    level = (row.get("level") or "").strip().upper()

    level_counts[level] += 1

    if not trace_id:
        rows_without_trace += 1
        continue

    flags = trace_flags.get(trace_id, 0)

    if level == "ERROR":
        flags |= 1
    elif level == "WARN":
        flags |= 2
    elif level != "INFO":
        flags |= 4

    trace_flags[trace_id] = flags

result = {
    "total_rows": total_rows,
    "distinct_tc_trace_id": len(trace_flags),
    "tc_trace_id_with_at_least_one_ERROR": sum(
        bool(flags & 1)
        for flags in trace_flags.values()
    ),
    "tc_trace_id_with_at_least_one_WARN": sum(
        bool(flags & 2)
        for flags in trace_flags.values()
    ),
    "tc_trace_id_with_only_INFO": sum(
        flags == 0
        for flags in trace_flags.values()
    ),
}

print(result)

[progresso] 100,000 linhas processadas
[progresso] 200,000 linhas processadas
[progresso] 300,000 linhas processadas
[progresso] 400,000 linhas processadas
[progresso] 500,000 linhas processadas
[progresso] 600,000 linhas processadas
[progresso] 700,000 linhas processadas
[progresso] 800,000 linhas processadas
[progresso] 900,000 linhas processadas
[progresso] 1,000,000 linhas processadas
[progresso] 1,100,000 linhas processadas
[progresso] 1,200,000 linhas processadas
[progresso] 1,300,000 linhas processadas
[progresso] 1,400,000 linhas processadas
[progresso] 1,500,000 linhas processadas
[progresso] 1,600,000 linhas processadas
[progresso] 1,700,000 linhas processadas
[progresso] 1,800,000 linhas processadas
[progresso] 1,900,000 linhas processadas
[progresso] 2,000,000 linhas processadas
[progresso] 2,100,000 linhas processadas
[progresso] 2,200,000 linhas processadas
[progresso] 2,300,000 linhas processadas
[progresso] 2,400,000 linhas processadas
[progresso] 2,500,000 linhas proce

In [10]:
total_rows = result["total_rows"]
distinct_tc_trace_id = result["distinct_tc_trace_id"]
tc_trace_id_with_at_least_one_ERROR = result["tc_trace_id_with_at_least_one_ERROR"]
tc_trace_id_with_at_least_one_WARN = result["tc_trace_id_with_at_least_one_WARN"]
tc_trace_id_with_only_INFO = result["tc_trace_id_with_only_INFO"]

percent_sample = 0.01

percent_sample_rows = int(total_rows * percent_sample)
percent_sample_distinct_tc_trace_id = int(distinct_tc_trace_id * percent_sample)
percent_sample_tc_trace_id_with_at_least_one_ERROR = int(tc_trace_id_with_at_least_one_ERROR * percent_sample)
percent_sample_tc_trace_id_with_at_least_one_WARN = int(tc_trace_id_with_at_least_one_WARN * percent_sample)
percent_sample_tc_trace_id_with_only_INFO = int(tc_trace_id_with_only_INFO * percent_sample)

sample_result = {
    "percent_sample": percent_sample,
    "percent_sample_rows": percent_sample_rows,
    "percent_sample_distinct_tc_trace_id": percent_sample_distinct_tc_trace_id,
    "percent_sample_tc_trace_id_with_at_least_one_ERROR": percent_sample_tc_trace_id_with_at_least_one_ERROR,
    "percent_sample_tc_trace_id_with_at_least_one_WARN": percent_sample_tc_trace_id_with_at_least_one_WARN,
    "percent_sample_tc_trace_id_with_only_INFO": percent_sample_tc_trace_id_with_only_INFO,
}

sample_result_json = json.dumps(sample_result, indent=4)
sample_result_path = Path("sample_result.json")
with sample_result_path.open("w", encoding="utf-8") as f:
    f.write(sample_result_json)